# Covariate-aware lesion–deficit substrate recovery

Classic lesion–symptom mapping (VLSM, SVR-LSM) and our two-arm VAE all try to recover the
neural **substrate** — which regions, when damaged, cause a deficit. But real deficits also
depend on **covariates** (age, education, …), and those covariates are often *confounded*
with the lesion itself — e.g. **older patients tend to have larger lesions**, so age
correlates with the deficit *and* with how much of any region is damaged. Ignoring it biases
the recovered substrate; the standard fix is to **control for the covariate** — nuisance
regression in VLSM, nuisance features in SVR-LSM, and a covariate head on the VAE.

This tutorial builds a covariate-conditioned deficit, runs each method **covariate-blind vs
covariate-aware**, and asks: *does controlling for the covariate recover the substrate
better?* The answer is more interesting than "always yes" — it hinges on how collinear the
covariate is with the lesion.

Everything runs in **Glasser-360 parcel space (2 mm)** on a shipped 512-lesion sample; the
model/baseline logic lives in importable modules (`covariate_deficits.py`, `baselines.py`,
`cov_vae.py`), so this notebook is narration + calls.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import baselines as B
from covariate_deficits import make_covariate_deficits, build_covariates, RULES
from cov_vae import fit_substrate_vae
from data_utils import load_episodes, load_atlas, to_volume

# Knobs (dial down for a faster run):
VAE_EPOCHS = 500     # VAE fit length per episode
N_PERM = 2000        # permutations for VLSM / SVR significance (see the p-value-floor note below)
SEED = 0

We run every method through one helper so blind vs aware is a single `cov=` argument.
Each method yields a **continuous score map** (for the confound-leakage diagnostic) and a
**binary mask** it chooses itself — VLSM/SVR by permutation significance, the VAE by its
calibrated `focal_mass_mask` readout.

In [ ]:
def recover(X, y, cov, *, n_perm=N_PERM, epochs=VAE_EPOCHS, seed=SEED):
    """Return {method: {'score': (K,), 'mask': (K,) bool}} for blind (cov=None) or aware."""
    vae_map = fit_substrate_vae(X, y, cov=cov, epochs=epochs, seed=seed)
    return {
        "vlsm": {"score": B.vlsm_stat(X, y, cov=cov),
                 "mask":  B.vlsm_binary(X, y, n_perm=n_perm, cov=cov, seed=seed)},
        "svr":  {"score": B.svr_lsm_weights(X, y, cov=cov),
                 "mask":  B.svr_lsm_binary(X, y, n_perm=n_perm, cov=cov, seed=seed)},
        "vae":  {"score": vae_map, "mask": B.focal_mass_mask(vae_map)},
    }

## 1. Data — parcel-space lesion sample

One **512-lesion sample per substrate** as parcel-overlap matrices `X` `(N=512, K=360)`
(fraction of each Glasser parcel lesioned), binary substrates `M` `(360,)`, and parcel
centroids (used by the spatially-structured deficit rules). Six NeuroQuery substrates,
~10 parcels each.

In [ ]:
ep = load_episodes("new_data")
X_all, M_all, centroids, names = ep["X_all"], ep["M_all"], ep["centroids"], ep["names"]
print("episodes X_all:", X_all.shape, " substrates M_all:", M_all.shape)
for i, n in enumerate(names):
    print(f"  [{i}] {n:32s} |M| = {int(M_all[i].sum())} parcels")

## 2. The covariate-conditioned deficit

For a chosen substrate, `make_covariate_deficits` builds the deficit. Per patient:

```
severity = g · clip(1 + β_int · age, 0)   +   β_main · age
           └ substrate × interaction ┘        └ main effect ┘
y = top-(target_rate · N) patients by severity        # positive rate held fixed
```

`g` is the (z-scored) substrate propensity for the rule. **age** is built so
`corr(age, lesion_volume) ≈ ρ` — the lesion-volume confound expressed through age; **sex**
is an inert nuisance. `β_main` is age's main effect, `β_int` the age × substrate
interaction. Four deficit rules: {coverage, bool_OR, hub_spoke, bool_AND}.

In [ ]:
SUB = names.index("dorsal_premotor")   # a representative substrate (all methods fire)
X, M = X_all[SUB], M_all[SUB]
deficits, cov = make_covariate_deficits(X, M, centroids, beta_main=0.5, beta_int=0.5,
                                        rho=0.5, seed=SEED)
age, volume = cov[:, 0], X.sum(1)
print(f"substrate: {names[SUB]}   corr(age, lesion volume) = "
      f"{np.corrcoef(age, volume)[0,1]:+.3f}   (target rho = 0.5)")
for r in RULES:
    pos = deficits[r] > 0.5
    print(f"  {r:10s}  n_pos={int(pos.sum()):3d}  mean age(+)= {age[pos].mean():+.2f} "
          f"vs age(-)= {age[~pos].mean():+.2f}")

Positives skew older (the **main effect**), and because age is tied to lesion volume, a
covariate-blind method will see non-substrate parcels light up just because big lesions
(older patients) touch them. That is the confound to control for.

## 3. Blind vs aware recovery

Each method exposes `cov=`: **VLSM** residualizes the deficit on `[1, C]`; **SVR-LSM**
appends standardized covariates as nuisance features (back-projecting only parcels); the
**VAE** adds a deficit head `scale·overlap·(1+Cγ)+Cβ+bias` with covariates entering the head
**only**, never the encoder, so the `E[p·w]` readout stays covariate-free.

We score with **binary Dice** — each method's *own* binarised mask vs the true substrate
(the deployment-relevant metric: no method is handed the true size) — plus **leakage**: how
much a method's off-target weight tracks the age confound (lower = less biased).

In [ ]:
RULE = "coverage"
y = deficits[RULE]
exposure = B.age_exposure(X, cov)
rec = {"blind": recover(X, y, None), "aware": recover(X, y, cov)}
print(f"substrate={names[SUB]}  rule={RULE}\n")
print(f"{'method':6s} {'Dice blind':>11s} {'Dice aware':>11s} {'leak blind':>11s} {'leak aware':>11s}")
for m in ("vlsm", "svr", "vae"):
    db = B.dice(rec['blind'][m]['mask'], M); da = B.dice(rec['aware'][m]['mask'], M)
    lb = B.leakage(rec['blind'][m]['score'], M, exposure)
    la = B.leakage(rec['aware'][m]['score'], M, exposure)
    print(f"{m:6s} {db:>11.3f} {da:>11.3f} {lb:>11.3f} {la:>11.3f}")

## 4. Visualising recovery — all three methods

Each method's **own self-thresholded mask** (exactly what binary Dice scores) vs the truth,
on one representative substrate, as an axial maximum-intensity projection. **Blue = truth,
red = recovered, purple = overlap** (blue-only = missed, red-only = false positive). Each
panel is annotated with its Dice and `n` = parcels it selected (true size |M| below). The
point binary Dice rewards is visible here: the VAE self-sizes near |M|, while the classical
significance thresholds over- or under-select. **This is one example substrate for spatial
intuition — do *not* rank the methods from it (per-substrate variance is large; e.g. this one
is unusually SVR-favourable). The method ranking is the summary bars in §5.**

In [ ]:
atlas = load_atlas("new_data")
k = int(M.sum())

def mask_mip(mask):
    mip = to_volume(np.asarray(mask, float), atlas).max(2).T[::-1]
    return np.ma.masked_where(mip == 0, mip)

gt_mip = to_volume(M, atlas).max(2).T[::-1]
gt_mip = np.ma.masked_where(gt_mip == 0, gt_mip)
methods = ["vlsm", "svr", "vae"]
fig, axes = plt.subplots(len(methods), 2, figsize=(7.5, 9.5))
for row, m in enumerate(methods):
    for col, mode in enumerate(["blind", "aware"]):
        ax = axes[row, col]; mask = rec[mode][m]["mask"]
        ax.imshow(gt_mip, cmap="Blues", vmin=0, vmax=1, alpha=0.55)
        ax.imshow(mask_mip(mask), cmap="Reds", vmin=0, vmax=1, alpha=0.6)
        ax.set_title(f"{m.upper()} {mode}   Dice={B.dice(mask, M):.2f}  n={int(np.sum(mask))}",
                     fontsize=10); ax.axis("off")
fig.suptitle(f"{names[SUB]} — self-thresholded recovery vs truth ({RULE});  |M| = {k}\n"
             "blue = truth   red = recovered   purple = overlap", fontsize=11)
plt.tight_layout(); plt.show()

## 5. Full benchmark — every substrate × rule, blind vs aware

We report **two** metrics, because they ask different questions and different methods win
each:

* **Dice@k** — Dice of each method's top-|M| ranked parcels. Pure *ranking / localization*
  quality (every method is handed the true size). This is where multivariate **SVR-LSM** is
  strongest — beating mass-univariate VLSM, as the literature expects.
* **Binary Dice** — each method's *own* thresholded mask vs truth. The *deployment* metric
  (no oracle size). It rewards the **VAE**, whose calibrated readout reliably self-sizes near
  |M|. Under the volume-collinear confound the linear correctors (VLSM residualization, SVR
  nuisance features) **mildly over-correct** — regressing out age also strips collinear
  substrate signal — so their *aware* Dice dips a little below *blind*; the VAE's covariate
  head controls without that penalty.

**On `N_PERM`.** The permutation p-value floor is `1/(N_PERM+1)`; for a per-parcel
FDR threshold `≈ q/K` to be reachable, the floor must sit below it. Voxel-space LSM
(10⁴–10⁵ tests) uses ~10 000 permutations (DeMarco & Turkeltaub 2018); here in **parcel
space (K=360)** the bar is far lower, so `N_PERM=2000` (floor 5×10⁻⁴) is ample. Too few
(e.g. 500) artificially **collapses VLSM's BH-FDR to empty masks** — an artifact, not
over-correction. SVR-LSM's max-statistic FWER is stable even at fewer permutations.
*(This is the slow cell; VLSM permutations are cheap, SVR's refit the SVR each time.)*

In [ ]:
def run_table(rho, n_perm=N_PERM, epochs=VAE_EPOCHS):
    ms = ("vlsm", "svr", "vae")
    mk = lambda: {m: {mo: {r: [] for r in RULES} for mo in ("blind", "aware")} for m in ms}
    dk, bn = mk(), mk()
    leak = {m: {"blind": [], "aware": []} for m in ms}
    for s in range(len(names)):
        Xs, Ms = X_all[s], M_all[s]
        defs, c = make_covariate_deficits(Xs, Ms, centroids, beta_main=0.5, beta_int=0.5,
                                          rho=rho, seed=SEED)
        exp = B.age_exposure(Xs, c)
        for r in RULES:
            ys = defs[r]
            for mode, cc in [("blind", None), ("aware", c)]:
                rc = recover(Xs, ys, cc, n_perm=n_perm, epochs=epochs)
                for m in ms:
                    dk[m][mode][r].append(B.dice_at_k(rc[m]["score"], Ms))
                    bn[m][mode][r].append(B.dice(rc[m]["mask"], Ms))
                    leak[m][mode].append(B.leakage(rc[m]["score"], Ms, exp))
    return ms, dk, bn, leak

def show(ms, tab, title):
    print(title + "\n" + f"{'rule':10s}" + "".join(f"{m+'_bl':>9s}{m+'_aw':>9s}" for m in ms))
    for r in RULES:
        print(f"{r:10s}" + "".join(f"{np.mean(tab[m]['blind'][r]):>9.3f}"
                                   f"{np.mean(tab[m]['aware'][r]):>9.3f}" for m in ms))
    print(f"{'MEAN':10s}" + "".join(
        f"{np.mean([np.mean(tab[m]['blind'][r]) for r in RULES]):>9.3f}"
        f"{np.mean([np.mean(tab[m]['aware'][r]) for r in RULES]):>9.3f}" for m in ms))

ms, dk, bn, leak = run_table(rho=0.5)
show(ms, dk, "Dice@k  (ranking; SVR-LSM strongest classical) — blind -> aware")
print()
show(ms, bn, "Binary Dice  (self-thresholded / deployment; VAE strongest) — blind -> aware")
print("\nleakage (mean; lower = less confound bias), blind -> aware")
for m in ms:
    print(f"  {m:5s} {np.mean(leak[m]['blind']):+.3f} -> {np.mean(leak[m]['aware']):+.3f}")

**Summary bars — mean over the 6 substrates (the headline).** Read the method ranking
here, not off the single-substrate glass-brain above (which has large per-substrate variance).
Each method, blind vs covariate-aware, on both metrics:

In [ ]:
def _m(tab, method, mode):
    return float(np.mean([np.mean(tab[method][mode][r] ) for r in RULES]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2)); x = np.arange(len(ms)); w = 0.38
for ax, tab, title in [(axes[0], dk, "Dice@k  (ranking)"),
                       (axes[1], bn, "Binary Dice  (deployment)")]:
    bl = [_m(tab, m, "blind") for m in ms]; aw = [_m(tab, m, "aware") for m in ms]
    ax.bar(x - w/2, bl, w, label="blind", color="#9aa7b5")
    ax.bar(x + w/2, aw, w, label="aware", color="#236a8f")
    for i, (b, a) in enumerate(zip(bl, aw)):
        ax.text(i - w/2, b + 0.006, f"{b:.2f}", ha="center", fontsize=8)
        ax.text(i + w/2, a + 0.006, f"{a:.2f}", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels([m.upper() for m in ms])
    ax.set_ylim(0, max(max(bl), max(aw)) * 1.25); ax.set_title(title)
    ax.legend(frameon=False, loc="upper right"); ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Mean substrate recovery over 6 substrates — blind vs covariate-aware")
plt.tight_layout(); plt.show()

Read together: **SVR-LSM localizes best** on the ranking metric (Dice@k) — the
multivariate method edging mass-univariate VLSM (and VAE-blind) by a hair, as the literature
says. **The VAE recovers best** on the deployment metric (binary Dice) by a clear margin,
because its calibrated readout self-sizes near the true substrate on every substrate, and it
is the **only method covariate-awareness helps** — its aware Dice rises on both metrics and
its confound leakage drops the most, while the linear correctors mildly over-correct (aware a
touch below blind) under the volume-collinear confound.

## 6. Why "aware" isn't always better — the ρ sweep

Controlling for a covariate reduces **bias**, but not necessarily **Dice**. When age is
*collinear* with the lesion (large ρ), regressing it out also strips genuine substrate signal
— the classic lesion-volume **over-correction**. We sweep ρ:

* **ρ = 0** — age drives the deficit but is independent of the lesion ⇒ no collinearity ⇒
  control should help.
* **ρ → 0.75** — strong volume confound ⇒ linear controls (VLSM/SVR) over-correct; the VAE's
  learned head degrades more gracefully.

*(3× the previous cell; reduce `n_perm`/`VAE_EPOCHS` if slow.)*

In [ ]:
rhos = [0.0, 0.5, 0.75]
sweep = {r: run_table(rho=r, n_perm=N_PERM)[2] for r in rhos}  # [2] = binary Dice
print(f"{'rho':>5s}" + "".join(f"{m+'_bl':>9s}{m+'_aw':>9s}" for m in ms)
      + "     (mean binary Dice, blind -> aware)")
for r in rhos:
    d = sweep[r]
    print(f"{r:>5.2f}" + "".join(
        f"{np.mean([np.mean(d[m]['blind'][rr]) for rr in RULES]):>9.3f}"
        f"{np.mean([np.mean(d[m]['aware'][rr]) for rr in RULES]):>9.3f}" for m in ms))

## 7. Takeaways

- **Different methods win different metrics — pick the one that matches your question.** On
  **Dice@k** (ranking / localization) the multivariate **SVR-LSM** is the strongest classical
  method, beating mass-univariate VLSM — the literature expectation. On **binary Dice**
  (self-thresholded, *deployable* — no oracle size) the **VAE** recovers best, because its
  calibrated readout self-sizes the mask while SVR-LSM's conservative permutation threshold
  under-selects on some substrates.
- **The VAE's covariate-aware head helps or holds** across ρ while reducing confound leakage —
  covariates enter the deficit head only, leaving the substrate readout clean.
- **Covariate control is a bias/variance trade, not a free win.** When age is independent of
  the lesion (ρ→0) there is no collinear signal to strip, so control tends to help; as age
  becomes collinear with lesion volume the linear correctors **over-correct** and lose Dice —
  the well-known lesion-volume-control pitfall — while the VAE's head degrades more gracefully.
- **Read Dice with leakage.** Dice measures recovery; leakage measures confound bias. Control
  reliably lowers leakage even where Dice doesn't move — and a near-empty mask has trivially
  low leakage, so read them together.